# 1. 프로젝트 폴더 구조 만들기 (src/envs, src/algorithms, scripts)

**“환경을 표현한다 → 상태 전이를 만든다 → Bellman update를 구현한다 → 실행 스크립트를 만든다”** 순서로 직접 쌓아가는 것입니다. 과제는 4×4 GridWorld를 사용합니다.

1. **먼저 프로젝트 뼈대만 만든다.** 처음부터 알고리즘을 쓰지 말고 역할별 파일을 나눕니다.

```text
assignment2-pytorch/
├── pyproject.toml
├── .gitignore
├── README.md
│
├── src/
│   └── connected_car_rl/
│       ├── __init__.py
│       ├── envs/
│       │   ├── __init__.py
│       │   └── gridworld.py
│       └── algorithms/
│           ├── __init__.py
│           └── dynamic_programming.py
│
├── scripts/
│   ├── policy_evaluation.py
│   └── value_iteration.py
│
└── tests/
    └── test_dynamic_programming.py
```

이 구조에서 `envs/`는 **문제가 어떻게 생겼는지**, `algorithms/`는 **그 문제를 어떻게 푸는지**, `scripts/`는 **실제로 무엇을 실행할지**를 담당합니다.

처음 손으로 구현할 때 가장 중요한 원칙은 `policy_evaluation.py`부터 시작하지 않는 것입니다. 아직 환경이 없기 때문입니다. 따라서 가장 먼저 `gridworld.py`를 완성합니다.

`src/connected_car_rl/envs/gridworld.py`를 열고 필요한 라이브러리부터 적습니다.

# 2. 문제 환경부터 만들기 gridworld.py

In [ ]:
from __future__ import annotations

# GridWorld의 설정값을 깔끔하게 묶기 위한 python 기능
from dataclasses import dataclass, field

# states, values, rewards, policy를 저장하기 위한 목적 (torch.Tensor 형태로)
import torch

In [ ]:
@dataclass
class GridWorld:
    rows: int = 4
    cols: int = 4
    initial_state: int = 0
    terminal_state: int = 15
    step_reward: float = -1.0
    device: torch.device | str = "cpu"
    dtype: torch.dtype = torch.float64

여기까지 작성하면 아직 GridWorld가 움직이지는 않습니다. 단지 “4×4 격자이고, 시작 상태는 0, terminal은 15이며, 한 번 이동할 때 보상은 -1이다”라는 설정만 저장한 상태입니다.

PyTorch는 상태 가치, 보상, 정책 등을 `torch.Tensor`로 저장하기 위해 사용합니다. `dataclass`는 GridWorld의 설정값을 깔끔하게 묶기 위한 Python 기능입니다.

action 정의: North, South, East, West 순서대로

0 -> North -> (-1, 0) \
1 -> South -> (+1, 0) \
2 -> East  -> (0, +1) \
3 -> West  -> (0, -1)

In [9]:
    action_deltas: tuple[tuple[int, int], ...] = field(
        default=((-1, 0), (1, 0), (0, 1), (0, -1)),
        init=False,
    )

In [ ]:
    def __post_init__(self) -> None:
        self.device = torch.device(self.device)

        self.num_states = self.rows * self.cols
        self.num_actions = len(self.action_deltas)

여기서부터가 환경 구현의 핵심입니다. 각 상태에서 각각의 행동을 했을 때 **다음 상태가 무엇인지 미리 표로 만들어두는 방식**을 사용합니다.

In [ ]:
        self.next_state = self._build_next_state_table()

아직 `_build_next_state_table()`을 만들지 않았으므로 잠시 그대로 둡니다.

다음으로 보상 표를 만듭니다.

In [ ]:
        self.reward = torch.full(
            (self.num_states, self.num_actions),
            self.step_reward,
            dtype=self.dtype,
            device=self.device,
        )

여기서 `reward`의 shape는 16 X 4, 행은 state 1개, 열은 action 1개

처음에는 모든 행동의 보상을 -1로 채웁니다.

```text
s0 : [-1, -1, -1, -1]
s1 : [-1, -1, -1, -1]
...
s15: [-1, -1, -1, -1]
```

하지만 `s15`는 terminal state이므로 그 상태에 도착한 뒤에는 더 이상 보상을 받아서는 안 됩니다. 따라서 다음을 추가합니다.

In [ ]:
        self.reward[self.terminal_state, :] = 0.0

이제 terminal state를 쉽게 구별할 수 있도록 mask도 만듭니다.

In [ ]:
        self.terminal_mask = torch.zeros(
            self.num_states,
            dtype=torch.bool,
            device=self.device,
        )

        self.terminal_mask[self.terminal_state] = True

여기까지 끝냈으면 이제 가장 중요한 `_build_next_state_table()`을 작성합니다.

먼저 함수와 빈 tensor를 만듭니다.

In [ ]:
    def _build_next_state_table(self) -> torch.Tensor:
        table = torch.empty(
            (self.num_states, self.num_actions),
            dtype=torch.long,
            device=self.device,
        )

이 `table` 역시 shape이 `[16, 4]`입니다. 차이는 reward가 아니라 **다음 state 번호**를 저장한다는 것입니다.

예를 들어 `s0`에서 South를 하면 `s4`이므로 대략 다음과 같은 정보가 들어갑니다.